In [14]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-03-03 16:21:01--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2025-03-03 16:21:01 (26.7 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [15]:
import torch
from torch import nn
from torch.nn import functional as F

In [16]:
# Hyper parameteres
train_size = 0.9
batch_size = 4
block_size = 8
emb_size = 32
dropout = 0.2
num_layers = 4
num_heads = 4

In [17]:
# Load input data
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()
  print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [18]:
# Define vocabulary set(Tokens), and vocabulary size
chars = set(text)
vocab_size = len(chars)


In [19]:
# Define decoding and encoding functions for vocabularies
stoi = {c:i for i,c in enumerate(chars)} # dictionary to convert a char to a number
itos = {i:c for i,c in enumerate(chars)} # dictionary to revert a number to  its equvalent char

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[id] for id in l)

In [20]:
# Convert data to torch tensor and create train and validation set
data = torch.tensor(encode(text))
n = int(train_size*len(data))
train = data[:n]
val = data[n:]



In [21]:
# create batch of samples
def get_batch(split):
  data = train if split == 'train' else val
  start_point = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[sp: sp+block_size] for sp in start_point])
  y = torch.stack([data[sp+1: sp+block_size+1] for sp in start_point])

  return x, y


In [ ]:
# Creat a class to build head
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.query = nn.Linear(emb_size, head_size, bias = False)  #Represents the current token’s intent—what it is looking for in other tokens.
    self.key = nn.Linear(emb_size, head_size, bias = False)    #Represents the information each token offers—how relevant it is to other tokens.
    self.value = nn.Linear(emb_size, head_size, bias = False)   #Contains the actual content/information of each token to be aggregated based on attention scores.
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) # We use register_buffer for tril because: It’s part of the model's computation but not a trainable parameter.
    self.dropout = nn.Dropout(dropout)                                           #self.tril is purely a sequence-level mask (not per head, not per embedding dimension).

  def forward(self, x):
    B,T,C = x.shape
    q = self.query(x) #(B,T,C) ----> (B, T, h)
    k = self.key(x)   #(B,T,C) ----> (B, T, h)
    wei = q @ k.transpose(-2,-1)*(self.head_size)**-0.5    #(B,T,h) @ (B,h,T) ----> (B, T, T)
    wei = wei.mask_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.somfmax(wei, dim = -1) # it is attention matrix
    self.dropout(wei) # by dropping out from wei certain tokens will not contribute to the final output during training
    v = self.value(x)

    out = wei @ v

    return out







In [23]:
class MultiHead(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads) ])
    self.proj = nn.Linear(num_heads*head_size, emb_size)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim = -1)
    out = self.dropout(self.proj(out))

    return out

# Each head captures different aspects of attention. self.proj allows the model to learn how to best combine the information from different heads instead of just concatenating them directly.
# Without self.proj, the next layers would receive raw concatenated head outputs, which might not be in the most useful format.

In [24]:
class FeedForward(nn.Module):
  def __init__(self, emb_size):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(emb_size, 4*emb_size),   #expansion-compression technique.
        nn.ReLU(),                            #The higher-dimensional space (4 * n_embd) lets the network learn more expressive representations.
        nn.Linear(4*emb_size, emb_size),    #This is inspired by deep learning heuristics where intermediate layers with more neurons can learn richer features.
        nn.Dropout(dropout),                #Empirically, using 4 * n_embd works well in Transformer models, which is why it has become the standard.

                      )

# Why No ReLU() After the Second Linear Layer?
# ReLU is used after the first Linear layer to introduce non-linearity, which helps the model learn complex patterns. However, we don't use ReLU after the second layer because:

# The second layer projects back to the original embedding size (n_embd), meaning it's acting more like a final transformation rather than a feature extractor.
# If we applied ReLU again, we’d force all negative values to zero, which might remove useful information.
# Typically, we want the final output to retain both positive and negative values for further processing.
# This design is commonly used in transformer-based architectures.


# Dropout is used as regularization to prevent overfitting. It randomly drops connections between neurons during training.

# If we applied dropout after the first Linear layer, we would drop information before it gets processed by the second layer, reducing the model’s learning capacity.
# Instead, we apply dropout at the end, after all transformations are done, so it only affects the final output before it is passed to the next layer.
# This approach helps prevent overfitting while keeping useful intermediate computations intact.

  def forward(self, x):
    return self.net(x)


In [25]:
class Block(nn.Module):
  def __init__(self, num_heads, emb_size):
    super().__init__()
    head_size = emb_size//num_heads
    self.sa = MultiHead(self, num_heads, head_size)
    self.ffwd = FeedForward(self, emb_size)
    self.ln1 = nn.LayerNorm(emb_size)
    self.nl2 = nn.LayerNorm(emb_size)

  def forward(self, x):
    out = out + self.sa(self.ln1(x))
    out = out + self.ffwd(self.ln1(out))

    return out

# LayerNorm (Layer Normalization) is a type of normalization layer used in deep learning, particularly in transformers.
# It normalizes the input across the feature dimension (not the batch dimension, like BatchNorm does).

# Standard Scaler in ML normalizes each feature across different samples (instances).
# LayerNorm normalizes activations in a neural network within each sample across features.

# Standard scaling helps traditional ML models converge better.
# LayerNorm improves stability in deep networks by preventing exploding/vanishing activations.

In [26]:
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, emb_size)
    self.position_embedding_table = nn.Embedding(vocab_size, emb_size)
    self.blocks = nn.sequential(*[Block(num_heads, emb_size) for _ in range(num_layers)])
    self.nl = nn.LayerNorm(emb_size)
    self.lm_head = nn.Linear(emb_size, vocab_size)


  def forward(self, idx, targets = None):
    B,T = idx.shape
    tok_emb = self.token_embedding_table(idx) # B,T,C
    pos_emb = self.position_embedding_table(torch.arange(T)) #B, T,C
    x = tok_emb + pos_emb  # B,T,C
    x = self.blocks(x)
    x = self.nl(x)
    logits = self.lm_head(x) # B, T, vocab_size

    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

      return loss, logits

  def generate(self, idx, max_new_token):

    for _ in range(max_new_token):
      idx_cond = idx[:,-block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:,-1,:]
      prob = F.softmax(logits, dim = -1)
      idx_next = torch.multinomial(prob, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx







